# ID2223 Lab Assignment 1 — Building and Evaluating CNNs on CIFAR-10

**AlexNet vs ResNet: capacity, regularization, robustness, adversarial examples and compute cost.**

This notebook follows the workflow from the Lab 1 notebook (`ID2223_Lab1.ipynb`): inspect the data, use a fixed seeded split, train with a `set_seed` / config dataclass setup, pick the checkpoint with the lowest validation cross-entropy, and open the test set only after all design decisions are frozen.

| Part | Content |
|---|---|
| 0 | Setup: device, run profile, artifact storage, reproducibility |
| Task 1 | Study and prepare CIFAR-10 |
| Task 2 | AlexNet and ResNet for CIFAR-10, baseline comparison, role of residual connections |
| Task 3 | Capacity (width/depth) and regularization (L2, dropout) |
| Task 4 | Data augmentation and robustness to corrupted images |
| Task 5 | Prediction analysis and adversarial examples |
| Task 6 | Benchmark and tradeoffs |
| Challenge 1 | Model compression (pruning, FP16, int8) |
| Challenge 2 | Generalization to unseen corruption families |

## How to reproduce

**Google Colab (full results used in the report).** Open the notebook in Colab, choose *Runtime → Change runtime type → T4 GPU*, and run all cells. The profile `full` is selected automatically on CUDA. Finished runs are stored in Google Drive (`MyDrive/kth-sml-lab1/full/`), so after a disconnect you can just *Run all* again: finished experiments are loaded instead of retrained.

**Local Mac (Apple Silicon).** From the project folder:
```bash
uv venv --python 3.12 .venv && source .venv/bin/activate
uv pip install torch torchvision numpy pandas matplotlib scikit-learn jupyter
jupyter lab   # open this notebook and run all cells
```
On MPS the profile `local` (shorter training budget) is selected. Set the environment variable `LAB_PROFILE=smoke` for a 2-minute end-to-end pipeline check, or `LAB_PROFILE=full` to force the full budget.

Each run writes `runs/<name>.json` (configuration, history, metrics, timing, hardware), `checkpoints/<name>.pt` and figures to `figures/` under the artifact directory.

## 0. Setup

In [1]:
import copy
import gzip
import io
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from torch import nn
from torchvision.datasets import CIFAR10

try:
    from IPython.display import display
except ImportError:  # running as a plain script
    display = print

IN_COLAB = "google.colab" in sys.modules
pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

print("Python:", platform.python_version(), "| PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Running in Colab:", IN_COLAB)

Python: 3.12.14 | PyTorch: 2.15.0 | torchvision: 0.29.2
Running in Colab: False


**Device.** CUDA (Colab T4) is preferred, then Apple MPS, then CPU. Mixed precision (FP16 autocast) is only used on CUDA; MPS and CPU train in FP32. The device name is stored with every run because all timing results are hardware dependent.

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    try:
        chip = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip()
    except Exception:
        chip = platform.machine()
    DEVICE_NAME = f"{chip} (MPS)"
else:
    device = torch.device("cpu")
    DEVICE_NAME = f"CPU ({platform.machine()})"

USE_AMP = device.type == "cuda"


def sync():
    """Wait for queued GPU work so wall-clock timings are honest."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def autocast():
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP)


print("Device:", device, "|", DEVICE_NAME, "| AMP:", USE_AMP)

Device: mps | Apple M4 (MPS) | AMP: False


**Run profile = compute budget.** Every model in one profile is trained with the *same* budget (epochs, batch size, schedule), which is what makes the comparisons fair.

| Profile | Used for | Training data | Epochs per run |
|---|---|---|---|
| `smoke` | pipeline check on any machine | 2,000 images | 2 |
| `local` | Apple M4 (MPS) | 45,000 images | 8 |
| `full` | Colab T4 — **results in the report** | 45,000 images | 15 |

In [3]:
PROFILES = {
    "smoke": dict(epochs=2, train_limit=2000, val_limit=1000, test_limit=1000, attack_n=64, pgd_steps=3, seed_repeats=1, finetune_epochs=1),
    "local": dict(epochs=8, train_limit=None, val_limit=None, test_limit=None, attack_n=500, pgd_steps=10, seed_repeats=1, finetune_epochs=1),
    "full": dict(epochs=15, train_limit=None, val_limit=None, test_limit=None, attack_n=1000, pgd_steps=10, seed_repeats=2, finetune_epochs=2),
}
PROFILE = os.environ.get("LAB_PROFILE") or {"cuda": "full", "mps": "local", "cpu": "smoke"}[device.type]
P = PROFILES[PROFILE]

FORCE_RETRAIN = False     # True = ignore stored runs and train everything again
RUN_CHALLENGES = True     # optional extensions at the end
USE_DRIVE = True          # Colab only: keep artifacts in Google Drive across sessions

print("Profile:", PROFILE, P)

Profile: local {'epochs': 8, 'train_limit': None, 'val_limit': None, 'test_limit': None, 'attack_n': 500, 'pgd_steps': 10, 'seed_repeats': 1, 'finetune_epochs': 1}


**Artifact storage.** On Colab, results go to Google Drive so they survive a disconnected runtime; locally they go to `artifacts/<profile>/` next to the notebook. Small JSON results and figures can be committed to Git; checkpoints (`*.pt`) are git-ignored.

In [4]:
if IN_COLAB and USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/kth-sml-lab1")
else:
    ARTIFACT_ROOT = Path("artifacts")
DATA_DIR = Path("/content/data") if IN_COLAB else Path("data")

ART = ARTIFACT_ROOT / PROFILE
RUN_DIR, CKPT_DIR, FIG_DIR, CACHE_DIR = (ART / d for d in ("runs", "checkpoints", "figures", "cache"))
for d in (RUN_DIR, CKPT_DIR, FIG_DIR, CACHE_DIR, DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


print("Artifacts:", ART.resolve())
print("Data:", DATA_DIR.resolve())

Artifacts: /Users/md.sakibulislam/Desktop/KTH/Period - 6/Scalable Machine Learning/Assignments/Assignment-1/artifacts/local
Data: /Users/md.sakibulislam/Desktop/KTH/Period - 6/Scalable Machine Learning/Assignments/Assignment-1/data


**Randomness control.** Separate seeds are used for each source of randomness, so that changing one experiment does not silently change another:

* `SPLIT_SEED` — the train/validation partition (same value as in Lab 1),
* `BASE_SEED` — weight initialization, mini-batch order, dropout masks and augmentation (one `torch.Generator` per run),
* `CORRUPTION_SEED` — the fixed corrupted evaluation sets,
* `ATTACK_SEED` — the random start of the PGD attack.

cuDNN autotuning is disabled and deterministic algorithms are requested. Bit-exact reproducibility is still only guaranteed on the same hardware and software versions: some GPU kernels (and MPS in general) are not deterministic, which is why Task 6 also repeats the baselines with different seeds.

In [5]:
SPLIT_SEED = 2223
BASE_SEED = 42
CORRUPTION_SEED = 7
ATTACK_SEED = 123

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if device.type == "mps":
        torch.mps.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed(BASE_SEED)

# Task 1. Study and prepare the data

### 1.1 Load CIFAR-10
CIFAR-10 ([Krizhevsky, 2009](https://www.cs.toronto.edu/~kriz/cifar.html)) is loaded with `torchvision.datasets.CIFAR10`, which downloads the official archive and verifies its MD5 checksum. We keep the raw `uint8` pixels and convert them to `[N, 3, 32, 32]` tensors (channels first, as PyTorch convolutions expect).

In [6]:
train_source = CIFAR10(DATA_DIR, train=True, download=True)
test_source = CIFAR10(DATA_DIR, train=False, download=True)
CLASSES = train_source.classes
NUM_CLASSES = len(CLASSES)

X_trainval = torch.from_numpy(train_source.data).permute(0, 3, 1, 2).contiguous()  # uint8 [50000, 3, 32, 32]
y_trainval = torch.tensor(train_source.targets, dtype=torch.long)
X_test_all = torch.from_numpy(test_source.data).permute(0, 3, 1, 2).contiguous()   # uint8 [10000, 3, 32, 32]
y_test_all = torch.tensor(test_source.targets, dtype=torch.long)

print("Provided training images:", tuple(X_trainval.shape), X_trainval.dtype)
print("Provided test images:    ", tuple(X_test_all.shape), X_test_all.dtype)
print("Pixel range:", int(X_trainval.min()), "to", int(X_trainval.max()))
print("Classes:", CLASSES)

assert X_trainval.shape == (50000, 3, 32, 32) and X_test_all.shape == (10000, 3, 32, 32)
assert set(y_trainval.unique().tolist()) == set(range(10))

Provided training images: (50000, 3, 32, 32) torch.uint8
Provided test images:     (10000, 3, 32, 32) torch.uint8
Pixel range: 0 to 255
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### 1.2 What does an example represent, and how are the classes distributed?
An example is a 32×32 RGB photograph (3,072 values in 0–255) of a single object, labelled with one of ten mutually exclusive classes: six animals and four vehicles. Objects are roughly centred, but vary in pose, scale, background and lighting. The resolution is very low, so fine details (for example, what separates a cat from a dog) are often only a few pixels.

In [7]:
class_counts = pd.DataFrame(
    {
        "provided train": np.bincount(y_trainval.numpy(), minlength=10),
        "provided test": np.bincount(y_test_all.numpy(), minlength=10),
    },
    index=CLASSES,
)
display(class_counts)

,provided train,provided test
airplane,5000,1000
automobile,5000,1000
bird,5000,1000
cat,5000,1000
deer,5000,1000
dog,5000,1000
frog,5000,1000
horse,5000,1000
ship,5000,1000
truck,5000,1000


In [8]:
fig, axes = plt.subplots(NUM_CLASSES, 8, figsize=(8, 10.5))
rng = np.random.default_rng(SPLIT_SEED)
for c in range(NUM_CLASSES):
    picks = rng.choice(np.flatnonzero(y_trainval.numpy() == c), size=8, replace=False)
    for j, idx in enumerate(picks):
        ax = axes[c, j]
        ax.imshow(X_trainval[idx].permute(1, 2, 0).numpy())
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        if j == 0:
            ax.set_ylabel(CLASSES[c], rotation=0, ha="right", va="center", fontsize=9)
fig.suptitle("Random training examples per class (ID chosen with SPLIT_SEED)")
fig.tight_layout()
savefig(fig, "t1_examples")
plt.show()

/var/folders/d_/jxkltryn62l6l_ktdcg78zzh0000gn/T/ipykernel_23857/1316801946.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Image statistics.** Per-channel brightness and contrast differ between classes (for example, ships and airplanes often have blue/bright backgrounds). A classifier can exploit such shortcuts, which matters later when we change colours or add noise.

In [9]:
x01 = X_trainval.float() / 255.0
per_image = pd.DataFrame(
    {
        "class": [CLASSES[i] for i in y_trainval.tolist()],
        "brightness": x01.mean(dim=(1, 2, 3)).numpy(),
        "contrast (pixel std)": x01.std(dim=(1, 2, 3)).numpy(),
        "R": x01[:, 0].mean(dim=(1, 2)).numpy(),
        "G": x01[:, 1].mean(dim=(1, 2)).numpy(),
        "B": x01[:, 2].mean(dim=(1, 2)).numpy(),
    }
)
display(per_image.groupby("class").mean().round(3))
del x01

,brightness,contrast (pixel std),R,G,B
class,,,,,
airplane,0.558,0.204,0.526,0.560,0.589
automobile,0.458,0.237,0.471,0.455,0.447
bird,0.468,0.182,0.489,0.491,0.424
cat,0.456,0.217,0.495,0.456,0.416
deer,0.438,0.179,0.472,0.465,0.378
dog,0.460,0.217,0.500,0.465,0.417
frog,0.418,0.187,0.470,0.438,0.345
horse,0.466,0.221,0.502,0.480,0.417
ship,0.523,0.216,0.490,0.525,0.555


### 1.3 How is the dataset split?
Following Lab 1, the provided **50,000 training images** are split with a stratified, seeded split into **45,000 training** and **5,000 validation** images (500 per class). The **10,000 test images** are kept for the final evaluation only; they are not used for checkpoint selection, hyperparameter choice or model selection. Original sample IDs are kept, so every prediction can be traced back to its image.

In [10]:
all_ids = np.arange(len(y_trainval))
train_ids, val_ids = train_test_split(
    all_ids, test_size=5000, stratify=y_trainval.numpy(), random_state=SPLIT_SEED
)
test_ids = np.arange(len(y_test_all))


def stratified_limit(ids, labels, limit):
    """Smaller stratified subset for the smoke profile (no-op when limit is None)."""
    if limit is None or limit >= len(ids):
        return ids
    keep, _ = train_test_split(ids, train_size=limit, stratify=labels[ids], random_state=SPLIT_SEED)
    return keep


train_ids = np.sort(stratified_limit(train_ids, y_trainval.numpy(), P["train_limit"]))
val_ids = np.sort(stratified_limit(val_ids, y_trainval.numpy(), P["val_limit"]))
test_ids = np.sort(stratified_limit(test_ids, y_test_all.numpy(), P["test_limit"]))

# A fixed subset of the training split, evaluated without augmentation or dropout, gives a
# deterministic training accuracy of the selected checkpoint (the running accuracy logged during an
# epoch is measured with augmentation/dropout switched on and with changing weights).
train_eval_ids = np.sort(np.random.default_rng(SPLIT_SEED).choice(train_ids, size=min(5000, len(train_ids)), replace=False))

assert not set(train_ids) & set(val_ids)
split_counts = pd.DataFrame(
    {
        "train": np.bincount(y_trainval.numpy()[train_ids], minlength=10),
        "validation": np.bincount(y_trainval.numpy()[val_ids], minlength=10),
        "test": np.bincount(y_test_all.numpy()[test_ids], minlength=10),
    },
    index=CLASSES,
)
split_counts.loc["total"] = split_counts.sum()
display(split_counts)

,train,validation,test
airplane,4500,500,1000
automobile,4500,500,1000
bird,4500,500,1000
cat,4500,500,1000
deer,4500,500,1000
dog,4500,500,1000
frog,4500,500,1000
horse,4500,500,1000
ship,4500,500,1000
truck,4500,500,1000


### 1.4 Which preprocessing is needed for the two architectures?
* **Scale to [0, 1].** A fixed conversion (divide by 255) with no learned statistics.
* **Per-channel standardization** $z = (x - \mu_c)/\sigma_c$, where $\mu_c, \sigma_c$ are computed **on the training split only** (as with the `StandardScaler` in Lab 1). This centres the inputs of the first convolution and makes one learning rate suitable for all channels.
* **No resizing.** The original AlexNet expects 227×227 inputs and ResNet expects 224×224. Upsampling 32×32 images by 7× adds no information and costs about 50× more compute, so instead both architectures are *adapted* to 32×32 inputs (Task 2). Both models therefore receive exactly the same input tensor.
* **Normalization inside the model.** Standardization is implemented as the first layer (`Normalize`) of every model, with $\mu, \sigma$ stored as buffers in the checkpoint. The models therefore take raw `[0, 1]` images. This makes saved checkpoints self-contained (the "preprocessing contract" from Lab 1 A6), and lets corruptions and adversarial perturbations be defined in pixel space with valid pixel bounds.
* **Data augmentation** is a training-time choice, studied separately in Task 4. Baselines in Tasks 2 and 3 are trained **without augmentation**, so that overfitting is visible.

In [11]:
def channel_stats(images_uint8, chunk=5000):
    total = torch.zeros(3, dtype=torch.float64)
    total_sq = torch.zeros(3, dtype=torch.float64)
    n = 0
    for start in range(0, len(images_uint8), chunk):
        x = images_uint8[start:start + chunk].to(torch.float64) / 255.0
        total += x.sum(dim=(0, 2, 3))
        total_sq += (x ** 2).sum(dim=(0, 2, 3))
        n += x.shape[0] * x.shape[2] * x.shape[3]
    mean = total / n
    std = (total_sq / n - mean ** 2).sqrt()
    return mean.float(), std.float()


CIFAR_MEAN, CIFAR_STD = channel_stats(X_trainval[train_ids])
print("Training-split mean (R, G, B):", CIFAR_MEAN.numpy().round(4))
print("Training-split std  (R, G, B):", CIFAR_STD.numpy().round(4))

Training-split mean (R, G, B): [0.4913 0.4822 0.4464]
Training-split std  (R, G, B): [0.247  0.2434 0.2615]


### 1.5 Reusable, device-resident data pipeline
CIFAR-10 is small (150 MB as `uint8`), so each split is moved to the accelerator **once**, and batches are produced by indexing on the device. This avoids the CPU `DataLoader` bottleneck on Colab (only 2 CPU cores), and keeps augmentation on the GPU. Shuffling uses a seeded `torch.Generator`, as in Lab 1.

In [12]:
class TensorData:
    """A split held on the device: uint8 images, labels and the original sample IDs."""

    def __init__(self, images_uint8, labels, ids, name):
        self.x = images_uint8.to(device)
        self.y = labels.to(device)
        self.ids = torch.as_tensor(np.asarray(ids))
        self.name = name

    def __len__(self):
        return len(self.ids)


def iterate(data, batch_size, generator=None):
    """Yield (uint8 images, labels, ids). Shuffled when a generator is given."""
    n = len(data)
    order = torch.randperm(n, generator=generator) if generator is not None else torch.arange(n)
    for start in range(0, n, batch_size):
        idx = order[start:start + batch_size]
        idx_dev = idx.to(device)
        yield data.x[idx_dev], data.y[idx_dev], data.ids[idx]


def to_float(x_uint8):
    return x_uint8.float().div_(255.0)


SPLITS = {
    "train": TensorData(X_trainval[train_ids], y_trainval[train_ids], train_ids, "train"),
    "train_eval": TensorData(X_trainval[train_eval_ids], y_trainval[train_eval_ids], train_eval_ids, "train_eval"),
    "val": TensorData(X_trainval[val_ids], y_trainval[val_ids], val_ids, "val"),
    "test": TensorData(X_test_all[test_ids], y_test_all[test_ids], test_ids, "test"),
}
for k, v in SPLITS.items():
    print(f"{k:>10}: {len(v):>6} images on {v.x.device}")

     train:  45000 images on mps:0
train_eval:   5000 images on mps:0
       val:   5000 images on mps:0
      test:  10000 images on mps:0


### 1.6 Training-time data augmentation (used in Task 4)
Three augmentation conditions are implemented on the GPU with the run's seeded generator:

| Condition | Transformations |
|---|---|
| `none` | none (original images) |
| `standard` | random crop (4-pixel reflection padding) + random horizontal flip |
| `strong` | `standard` + with p = 0.5 a random rotation (±15°) and shear (±10°), + with p = 0.5 Gaussian noise (σ ~ U(0, 0.06)) |

Horizontal flips and small translations keep the label for every CIFAR-10 class. A vertical flip would not (an upside-down ship or horse is not a natural image), so it is not used.

In [13]:
def affine_batch(x, angle_deg, shear_deg=None, scale=None, padding_mode="reflection"):
    """Rotate / shear / scale each image of a [B, 3, H, W] batch around its centre (bilinear)."""
    B = x.shape[0]
    angle = torch.deg2rad(angle_deg.float())
    shear = torch.deg2rad(shear_deg.float()) if shear_deg is not None else torch.zeros(B)
    scale = scale.float() if scale is not None else torch.ones(B)
    cos, sin = torch.cos(angle), torch.sin(angle)
    rot = torch.stack([torch.stack([cos, -sin], -1), torch.stack([sin, cos], -1)], -2)          # [B, 2, 2]
    shr = torch.stack([torch.stack([torch.ones(B), torch.tan(shear)], -1),
                       torch.stack([torch.zeros(B), torch.ones(B)], -1)], -2)
    A = (rot @ shr) / scale[:, None, None]       # maps output coordinates to input coordinates
    theta = torch.cat([A, torch.zeros(B, 2, 1)], dim=2).to(x.device, x.dtype)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode=padding_mode, align_corners=False)


def random_crop_flip(x, gen, pad=4):
    B, C, H, W = x.shape
    padded = F.pad(x, (pad, pad, pad, pad), mode="reflect")
    dy = torch.randint(0, 2 * pad + 1, (B,), generator=gen).to(x.device)
    dx = torch.randint(0, 2 * pad + 1, (B,), generator=gen).to(x.device)
    rows = dy[:, None] + torch.arange(H, device=x.device)[None, :]
    cols = dx[:, None] + torch.arange(W, device=x.device)[None, :]
    out = padded[
        torch.arange(B, device=x.device)[:, None, None, None],
        torch.arange(C, device=x.device)[None, :, None, None],
        rows[:, None, :, None],
        cols[:, None, None, :],
    ]
    flip = (torch.rand(B, generator=gen) < 0.5).to(x.device)
    return torch.where(flip[:, None, None, None], out.flip(3), out)


def augment(x, kind, gen):
    """x: float batch in [0, 1] on the device. gen: CPU generator owned by the run."""
    if kind == "none":
        return x
    x = random_crop_flip(x, gen)
    if kind == "strong":
        B = x.shape[0]
        apply = (torch.rand(B, generator=gen) < 0.5).float()
        angle = (torch.rand(B, generator=gen) * 2 - 1) * 15 * apply
        shear = (torch.rand(B, generator=gen) * 2 - 1) * 10 * apply
        x = affine_batch(x, angle, shear)
        sigma = torch.rand(B, generator=gen) * 0.06 * (torch.rand(B, generator=gen) < 0.5).float()
        noise = torch.randn(x.shape, generator=gen) * sigma[:, None, None, None]
        x = (x + noise.to(x.device)).clamp_(0, 1)
    elif kind != "standard":
        raise ValueError(kind)
    return x


demo_gen = torch.Generator().manual_seed(0)
demo = to_float(SPLITS["train"].x[:6])
rows = {"original": demo, "standard": augment(demo, "standard", demo_gen), "strong": augment(demo, "strong", demo_gen)}
fig, axes = plt.subplots(3, 6, figsize=(9, 4.8))
for r, (title, batch) in enumerate(rows.items()):
    for j in range(6):
        axes[r, j].imshow(batch[j].permute(1, 2, 0).cpu().numpy())
        axes[r, j].axis("off")
    axes[r, 0].set_title(title, loc="left", fontsize=9)
fig.suptitle("Augmentation conditions (same six training images)")
fig.tight_layout()
savefig(fig, "t1_augmentations")
plt.show()

/var/folders/d_/jxkltryn62l6l_ktdcg78zzh0000gn/T/ipykernel_23857/4033865590.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
